# 🏔️ De prompt a producto
**Taller de 45 min: construye un prototipo de IA con la API de Gemini.**

Al final tendrás un asistente que recomienda **rutas turísticas del Valle del Mantaro**, con una interfaz y un link público que puedes abrir desde tu celular. Después lo adaptas a un problema tuyo.

| Checkpoint | Qué haces | Tiempo |
|---|---|---|
| **1** | Tu primera llamada a la API | 10-20 min |
| **2** | Del prompt a la lógica del producto (JSON + validaciones) | 20-32 min |
| **3** | Interfaz con Gradio y link público | 32-40 min |

### Antes de empezar (una sola vez)
1. Crea tu API key en [Google AI Studio](https://aistudio.google.com/apikey).
2. En Colab abre 🔑 **Secretos** (panel izquierdo), crea uno llamado `GEMINI_API_KEY`, pega tu key y activa **Acceso desde el notebook**.
3. Ejecuta la celda de abajo con ▶️. Tarda ~1 minuto. Si sale un aviso rojo de pip sobre "dependency conflicts", **ignóralo**: lo importante es ver ✅ al final.

> 🆘 **¿Sin internet o sin key?** En la celda de preparación cambia `USE_MOCK = False` por `USE_MOCK = True`. El taller corre con respuestas precargadas y todo el código es el mismo.
>
> 🧭 **¿Te atrasaste?** Abre la rama del checkpoint anterior desde el README del repositorio: ya viene resuelto.

In [ ]:
# ⚙️ Preparación: ejecútala una vez y sigue
!pip install -q google-genai gradio   # si sale un aviso rojo de "dependency conflicts", ignóralo

import os, json, subprocess

REPO = "https://github.com/luisverabyte/taller-prompt-a-producto"
if not os.path.exists("data/lugares.json"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "taller"], check=True)
    os.chdir("taller")

USE_MOCK = False   # True = sin internet / sin API key (respuestas precargadas)
MODEL = "gemini-3.8-flash"

try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    API_KEY = os.environ.get("GEMINI_API_KEY")

from google import genai
from google.genai import types

if USE_MOCK:
    from mock_client import MockClient
    client = MockClient()
    print("🔌 Modo SIN CONEXIÓN: usando respuestas precargadas")
elif not API_KEY:
    raise RuntimeError(
        "No encontré tu API key. Revisa que el secreto se llame GEMINI_API_KEY y que "
        "tenga activado el acceso desde el notebook. Si no tienes key o internet, "
        "pon USE_MOCK = True."
    )
else:
    client = genai.Client(api_key=API_KEY)
    print("✅ Listo, conectado con", MODEL)

---
## ✅ Checkpoint 1: tu primera llamada a la API
Un *request* tiene tres partes: el **modelo**, el **contenido** (tu prompt) y la **respuesta**. Eso es todo.

Cuando salga un texto abajo, levanta la mano o pega el sticker **verde** 🟢.

In [ ]:
respuesta = client.models.generate_content(
    model=MODEL,
    contents="Salúdame en una frase como si fueras una guía de turismo de Huancayo.",
)
print(respuesta.text)

---
## 🧠 Checkpoint 2: del prompt a la lógica del producto
Un chat responde texto libre. Un **producto** necesita una respuesta con forma fija, reglas y control de errores.

Vamos a construir tres cosas:
1. **Un esquema** (la forma del JSON que esperamos).
2. **Un system prompt** (el rol y las reglas del asistente).
3. **Validaciones** (revisar que la respuesta cumple antes de mostrarla).

**Tips de producto antes de escribir código:**
- **Usuario:** ¿para quién es? (Aquí: un visitante que llega a Huancayo con pocos días y un presupuesto.)
- **Caso de uso mínimo:** ¿cuál es la sola cosa que debe hacer bien? (Armar una ruta realista.)
- **Métrica:** ¿cómo sabes que funciona? (Por ejemplo: el % de rutas que pasan la validación a la primera.)

In [ ]:
from typing import Optional
from pydantic import BaseModel, ValidationError

with open("data/lugares.json", encoding="utf-8") as f:
    LUGARES = json.load(f)["lugares"]
NOMBRES = {l["nombre"] for l in LUGARES}
print(len(LUGARES), "lugares en el catálogo")


class Parada(BaseModel):
    nombre: str
    distrito: str
    horas: float
    costo_soles: int
    tip: str


class Dia(BaseModel):
    dia: int
    titulo: str
    paradas: list[Parada]


class Ruta(BaseModel):
    resumen: str
    dias: list[Dia]
    costo_total_soles: int
    advertencia: Optional[str] = None

In [ ]:
SYSTEM_PROMPT = f"""Eres una guía de turismo local del Valle del Mantaro (Huancayo, Perú).
Armas rutas para visitantes usando SOLO los lugares del catálogo.

Reglas:
1. Usa únicamente lugares del catálogo y copia su nombre exacto.
2. Cada día suma máximo 8 horas de visitas.
3. La suma de costo_soles de todas las paradas debe ser menor o igual al presupuesto.
4. Prioriza los intereses del visitante, pero mezcla al menos dos categorías.
5. Si los días o el presupuesto no alcanzan, dilo en "advertencia" en vez de inventar.
6. Cada parada lleva un tip corto y práctico.

Catálogo:
{json.dumps(LUGARES, ensure_ascii=False)}
"""
print(SYSTEM_PROMPT[:600], "...")

In [ ]:
def validar_ruta(ruta: Ruta, dias: int, presupuesto: int) -> list[str]:
    """Devuelve la lista de problemas. Lista vacía = la ruta es válida."""
    problemas = []
    if len(ruta.dias) != dias:
        problemas.append(f"Se pidieron {dias} días y la ruta tiene {len(ruta.dias)}.")
    for d in ruta.dias:
        for p in d.paradas:
            if p.nombre not in NOMBRES:
                problemas.append(f"El lugar {p.nombre!r} no está en el catálogo.")
        horas = sum(p.horas for p in d.paradas)
        if horas > 8:
            problemas.append(f"El día {d.dia} suma {horas} horas (máximo 8).")
    total = sum(p.costo_soles for d in ruta.dias for p in d.paradas)
    if total > presupuesto:
        problemas.append(f"El costo es S/ {total} y el presupuesto es S/ {presupuesto}.")
    return problemas

In [ ]:
def crear_ruta(dias, presupuesto, intereses, grupo, intentos=3):
    """Pide una ruta al modelo, la valida y reintenta pasando los errores."""
    pedido = (
        f"Días: {dias}\n"
        f"Presupuesto máximo por persona: S/ {presupuesto}\n"
        f"Intereses: {', '.join(intereses) or 'cualquiera'}\n"
        f"Viaja: {grupo}"
    )
    ruta, problemas = None, []
    for intento in range(1, intentos + 1):
        contenido = pedido
        if problemas:
            contenido += "\n\nCorrige estos problemas de tu respuesta anterior:\n- " + "\n- ".join(problemas)
        respuesta = client.models.generate_content(
            model=MODEL,
            contents=contenido,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                response_mime_type="application/json",
                response_schema=Ruta,
                temperature=0.4,
            ),
        )
        try:
            ruta = Ruta.model_validate_json(respuesta.text)
        except ValidationError:
            problemas = ["La respuesta no cumple el esquema JSON."]
            continue
        problemas = validar_ruta(ruta, dias, presupuesto)
        if not problemas:
            print(f"✅ Ruta válida en el intento {intento}")
            return ruta, []
        print(f"⚠️ Intento {intento}: {len(problemas)} problema(s), reintentando...")
    return ruta, problemas

In [ ]:
ruta, problemas = crear_ruta(
    dias=2,
    presupuesto=150,
    intereses=["artesanía", "cultura"],
    grupo="pareja",
)
if ruta:
    print(ruta.model_dump_json(indent=2))
print("Problemas:", problemas or "ninguno ✅")

### 🎯 Micro-reto: llévalo a tu problema real
Cambia **solo estas 3 variables** y adapta el asistente a turismo, agro, educación, comercio o lo que hagas en tu trabajo.
Piensa en el **usuario**, el **caso de uso mínimo** y **cómo medirías** si funciona.

In [ ]:
class Respuesta(BaseModel):
    titulo: str
    pasos: list[str]
    por_que: str
    advertencia: Optional[str] = None


# 👇 Cambia estas 3 variables
ROL = "Eres un asesor agrícola que ayuda a pequeños productores del Valle del Mantaro."
DATOS = "Cultivos de la zona: papa, maca, maíz, alcachofa. Heladas fuertes entre junio y agosto."
PREGUNTA = "Quiero sembrar papa en julio, ¿qué debo tener en cuenta?"


def consultar(rol, datos, pregunta):
    respuesta = client.models.generate_content(
        model=MODEL,
        contents=pregunta,
        config=types.GenerateContentConfig(
            system_instruction=f"{rol}\nUsa solo estos datos y no inventes otros:\n{datos}",
            response_mime_type="application/json",
            response_schema=Respuesta,
            temperature=0.4,
        ),
    )
    return Respuesta.model_validate_json(respuesta.text)


r = consultar(ROL, DATOS, PREGUNTA)
print(r.titulo)
for i, paso in enumerate(r.pasos, 1):
    print(f"{i}. {paso}")
print("Por qué:", r.por_que)

---
## 🖥️ Checkpoint 3: interfaz y compartir
Una API no es un producto hasta que alguien que no programa puede usarla. Con **Gradio** envolvemos la función `crear_ruta` en una interfaz web y obtenemos un **link público** para abrirlo desde el celular.

Cuando tengas tu link, súbelo al muro compartido que se proyecta en pantalla. 🔗

In [ ]:
import html


def ruta_a_html(ruta, problemas):
    """Convierte la ruta en tarjetas HTML (esto es solo presentación)."""
    if ruta is None:
        return "<p>😕 No pude armar una ruta. Intenta de nuevo.</p>"
    e = html.escape
    out = [f"<h3>{e(ruta.resumen)}</h3>",
           f"<p><b>Costo estimado:</b> S/ {ruta.costo_total_soles} por persona</p>"]
    for d in ruta.dias:
        out.append(f"<div style=\"border:1px solid #8884;border-radius:12px;padding:12px;margin:10px 0\">"
                   f"<h4>Día {d.dia}: {e(d.titulo)}</h4><ul>")
        for p in d.paradas:
            out.append(f"<li><b>{e(p.nombre)}</b> · {e(p.distrito)} · {p.horas} h · S/ {p.costo_soles}"
                       f"<br><i>💡 {e(p.tip)}</i></li>")
        out.append("</ul></div>")
    avisos = ([ruta.advertencia] if ruta.advertencia else []) + problemas
    for a in avisos:
        out.append(f"<p>⚠️ {e(a)}</p>")
    return "".join(out)

In [ ]:
import gradio as gr

INTERESES = ["cultura", "naturaleza", "gastronomía", "artesanía", "aventura", "familia"]


def planificar(dias, presupuesto, intereses, grupo):
    ruta, problemas = crear_ruta(int(dias), int(presupuesto), intereses, grupo)
    return ruta_a_html(ruta, problemas)


# TODO: arma la interfaz con gr.Blocks. Necesitas:
#   - entradas: gr.Slider (días 1-3), gr.Number (presupuesto), gr.CheckboxGroup(INTERESES), gr.Radio (grupo)
#   - un gr.Button y una salida gr.HTML()
#   - conectar con: boton.click(planificar, [entradas...], salida)
#   - lanzarla con: demo.launch(share=True)   → te da el link público

---
## 🚀 Y ahora, ¿cómo pasa de prototipo a producto?
- **Evaluación:** arma 10 casos de prueba y mide cuántos pasan tu validación.
- **Costos:** calcula cuánto cuesta cada consulta y cuántas harán tus usuarios.
- **Privacidad:** no mandes datos personales a la API si no es necesario.
- **Feedback:** pon el link en manos de 5 usuarios reales y escucha qué no entienden.

Sube tu versión mejorada y preséntala en un próximo meetup con **Prisma Latam** y **GDG Huancayo**. 📸